# Limpieza de datos – BAI Dataset

Este notebook ejecuta la ingesta de datos, validaciones, y procesamiento para el BAI dataset.

Pasos:
- Cargar data cruda
- Estandarizar datos base
- Revisar estructura
- Parsear respuestas
- Expandir los items del cuestionario BAI a columnas
- Preparar dataset final
- Validar dataset
- Almacenar dataset limpio

### Setup del proyecto

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

### Imports

In [2]:
from pathlib import Path

from src.config import BAI_DATA_PATH, DATA_PROCESSED
from src.data_loader import load_excel
from src.preprocessing import (
    expand_bai_answers,
    parse_answers_column,
    prepare_bai_dataset,
    standardize_bai_data,
    validate_bai_dataset,
)

### Cargar data cruda

In [3]:
df_raw = load_excel(BAI_DATA_PATH)
print("Shape original:", df_raw.shape)
print("Columnas:", df_raw.columns.tolist())
df_raw.drop(columns=['email', 'name']).head()

Shape original: (113, 9)
Columnas: ['id', 'age', 'answers', 'category', 'date', 'email', 'gender', 'name', 'totalScore']


,id,age,answers,category,date,gender,totalScore
0,024ccc066e5f36803f8861b9ee3d94888e5370faed41a9...,24,"[1, 2, 0, 0, 2, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, ...","""low""","July 22, 2025 at 12:45:35 PM UTC-6","""M""",11
1,02f2a803fa9512cbee77681b62aec389336d01bf56e54b...,19,"[2, 3, 0, 2, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, ...","""low""","July 23, 2025 at 9:17:01 AM UTC-6","""M""",15
2,044a25cf19eab34fffd83bdd006ce8338150351ac18ebf...,23,"[2, 2, 0, 2, 2, 0, 0, 0, 2, 2, 0, 0, 0, 1, 0, ...","""low""","July 22, 2025 at 12:49:46 PM UTC-6","""M""",16
3,05590f956566724748711d0b68f1b44f3fabfc87cab7cf...,25,"[0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...","""low""","July 22, 2025 at 12:40:20 PM UTC-6","""M""",4
4,0ba67bab767aadf47f4be2743015846e30a32459939c76...,23,"[0, 1, 0, 2, 2, 0, 0, 2, 0, 1, 0, 0, 0, 0, 0, ...","""low""","July 22, 2025 at 1:18:40 PM UTC-6","""M""",10


### Limpieza de datos

In [4]:
rows_before_standardization = len(df_raw)
df_standardized = standardize_bai_data(df_raw)
rows_after_standardization = len(df_standardized)

print("Filas originales:", rows_before_standardization)
print("Filas después de limpieza estándar:", rows_after_standardization)
print("Filas eliminadas por faltantes o duplicados:", rows_before_standardization - rows_after_standardization)
print(df_standardized[["category", "gender", "date"]].head())

Filas originales: 113
Filas después de limpieza estándar: 113
Filas eliminadas por faltantes o duplicados: 0
  category gender                                date
0      LOW      M  July 22, 2025 at 12:45:35 PM UTC-6
1      LOW      M   July 23, 2025 at 9:17:01 AM UTC-6
2      LOW      M  July 22, 2025 at 12:49:46 PM UTC-6
3      LOW      M  July 22, 2025 at 12:40:20 PM UTC-6
4      LOW      M   July 22, 2025 at 1:18:40 PM UTC-6


### Revisar estructura de `answers`

In [5]:
print("Tipo de dato en answers antes del parseo:", type(df_standardized.loc[0, "answers"]).__name__)
print("Ejemplo de answers crudo:", df_standardized.loc[0, "answers"])

Tipo de dato en answers antes del parseo: str
Ejemplo de answers crudo: [1, 2, 0, 0, 2, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2]


### Parsear y expandir respuestas BAI

In [6]:
df_parsed = parse_answers_column(df_standardized, column="answers")

print("Tipo de dato en answers despues del parseo:", type(df_parsed.loc[0, "answers"]).__name__)
print("Longitud del arreglo answers en la primera fila:", len(df_parsed.loc[0, "answers"]))
print("Ejemplo parseado:", df_parsed.loc[0, "answers"])

Tipo de dato en answers despues del parseo: list
Longitud del arreglo answers en la primera fila: 21
Ejemplo parseado: [1, 2, 0, 0, 2, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2]


In [7]:
df_clean = expand_bai_answers(df_parsed, column="answers")

bai_columns = [f"BAI_{i}" for i in range(1, 22)]
print("Columnas BAI creadas:", bai_columns)
print(df_clean[["id", "answers", *bai_columns]].head())

Columnas BAI creadas: ['BAI_1', 'BAI_2', 'BAI_3', 'BAI_4', 'BAI_5', 'BAI_6', 'BAI_7', 'BAI_8', 'BAI_9', 'BAI_10', 'BAI_11', 'BAI_12', 'BAI_13', 'BAI_14', 'BAI_15', 'BAI_16', 'BAI_17', 'BAI_18', 'BAI_19', 'BAI_20', 'BAI_21']
                                                  id  \
0  024ccc066e5f36803f8861b9ee3d94888e5370faed41a9...   
1  02f2a803fa9512cbee77681b62aec389336d01bf56e54b...   
2  044a25cf19eab34fffd83bdd006ce8338150351ac18ebf...   
3  05590f956566724748711d0b68f1b44f3fabfc87cab7cf...   
4  0ba67bab767aadf47f4be2743015846e30a32459939c76...   

                                             answers  BAI_1  BAI_2  BAI_3  \
0  [1, 2, 0, 0, 2, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, ...      1      2      0   
1  [2, 3, 0, 2, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, ...      2      3      0   
2  [2, 2, 0, 2, 2, 0, 0, 0, 2, 2, 0, 0, 0, 1, 0, ...      2      2      0   
3  [0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...      0      0      0   
4  [0, 1, 0, 2, 2, 0, 0, 2, 0, 1, 0, 0, 0, 0, 0, ...  

### Preparar dataset final

In [8]:
df_final = prepare_bai_dataset(df_clean)

print("Columnas eliminadas del dataset final: ['answers', 'email', 'name', 'date']")
print("Shape final:", df_final.shape)
print(df_final.head())

Columnas eliminadas del dataset final: ['answers', 'email', 'name', 'date']
Shape final: (113, 26)
                                                  id  age category gender  \
0  024ccc066e5f36803f8861b9ee3d94888e5370faed41a9...   24      LOW      M   
1  02f2a803fa9512cbee77681b62aec389336d01bf56e54b...   19      LOW      M   
2  044a25cf19eab34fffd83bdd006ce8338150351ac18ebf...   23      LOW      M   
3  05590f956566724748711d0b68f1b44f3fabfc87cab7cf...   25      LOW      M   
4  0ba67bab767aadf47f4be2743015846e30a32459939c76...   23      LOW      M   

   totalScore  BAI_1  BAI_2  BAI_3  BAI_4  BAI_5  ...  BAI_12  BAI_13  BAI_14  \
0          11      1      2      0      0      2  ...       0       0       0   
1          15      2      3      0      2      1  ...       0       0       0   
2          16      2      2      0      2      2  ...       0       0       1   
3           4      0      0      0      1      1  ...       0       0       0   
4          10      0      1      

### Validar y guardar dataset procesado

In [9]:
validate_bai_dataset(df_final)

output_path = Path(DATA_PROCESSED)
output_path.parent.mkdir(parents=True, exist_ok=True)
df_final.to_excel(output_path, index=False)

print("Validacion completada correctamente.")
## print("Archivo guardado en: ", output_path)

Validacion completada correctamente.
